In this notebook, we are going to load some of the data that we processed in the Condensed Notebook and perform across-trial analysis of the power spectra using FOOOF, an iterative peak-detection and PSD 1/f fitting algorithm. 

Also provide options for plotting the frequency-wide and session-wide PSD by region.

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import mne
import mne_connectivity
from mne_connectivity import phase_slope_index, seed_target_indices, spectral_connectivity_epochs
from glob import glob
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from scipy.stats import zscore, linregress, ttest_ind, ttest_rel, ttest_1samp
import pandas as pd
from mne.preprocessing.bads import _find_outliers
import fooof
# Import plotting functions
from fooof.plts.spectra import plot_spectrum, plot_spectra
# Import the FOOOF object
from fooof import FOOOF
from fooof import FOOOFGroup
import os 
import joblib
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pickle
from tqdm import tqdm
from IPython.display import clear_output
import warnings 
from ast import literal_eval


# I only want to see warnings once
warnings.filterwarnings('ignore')

from joblib import delayed, Parallel, delayed
from statsmodels.stats import multitest

from neurodsp.plts.spectral import plot_power_spectra


Note: If you have installed this package in editable form on Minerva, you must append the local path! This is because Minerva requires that you point your package installs away from the local directory for space reasons, but editable packages have to be installed locally.

In [ ]:
import sys
sys.path.append('/Users/salmanqasim/Documents/GitRepos/LFPAnalysis')
from LFPAnalysis import lfp_preprocess_utils, sync_utils, analysis_utils, nlx_utils, oscillation_utils, statistics_utils


Before running the following, you need to: 

1. Load relevant behavioral data produced from your proprietary behavioral analyses.
1. Load neural epochs of behavioral interest produced from the Condensed Notebook.

## First, get the behavior data

This is from another notebook with a kernel and environment specifically for beahvior

In [ ]:
base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site=='UI']

subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]


days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')

## Set some plotting parameters

In [ ]:
from plot_config import set_plot_style
set_plot_style()

In [ ]:
# chan = 'raglt2-raglt3'
# epo_spectrum = mne_data_reref.copy().pick([chan]).compute_psd(
#     method='multitaper',
#     fmin=2,
#     fmax=200,
#     tmin=0,
#     tmax=1.5,
#     verbose=False
# )

In [ ]:
# epo_spectrum.plot()
# sns.despine()

In [ ]:
day = 'Day1'

FOOOF_kwargs = {'peak_width_limits': [1, 8], 
               'min_peak_height': 0.1,
                'peak_threshold': 3.,
                'max_n_peaks': 6,
               'freq_range': [2, 50]}

analysis_evs = {'feedback_start': [0, 1.5],
                'cue_on': [0, 1]}

for event in analysis_evs.keys():
    for subj in subj_ids:
        chan_dfs = []
        load_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/neural/{day}'
        save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}'
        mne_data_reref = mne.read_epochs(f'{load_path}/{event}-epo.fif', preload=True)

        # # NAN out the bad data
        # # THE following will now LOAD in dataframes that indicate IED and artifact time points in your data
        # IED_df = pd.read_csv(f'{load_path}/{event}_IED_df.csv') 
        # artifact_df = pd.read_csv(f'{load_path}/{event}_artifact_df.csv') 

        # # Now, let's iterate through each channel, and each ied/artifact, and NaN 100 ms before and after these timepoints
        # for ch_ix, ch_name in enumerate(mne_data_reref.ch_names): 
        #     ied_ev_list = IED_df[ch_name].dropna().index.tolist()
        #     artifact_ev_list = artifact_df[ch_name].dropna().index.tolist() 
        #     for ev_ in ied_ev_list: 
        #         for ied_ in literal_eval(IED_df[ch_name].iloc[ev_]):
        #             # remove 100 ms before 
        #             ev_ix_start = np.max([0, np.floor((ied_- 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
        #             # remove 100 ms after
        #             ev_ix_end = np.min([mne_data_reref._data.shape[-1], np.ceil((ied_ + 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
        #             mne_data_reref._data[ev_, ch_ix, ev_ix_start:ev_ix_end] = np.nan
        #     for ev_ in artifact_ev_list: 
        #         for artifact_ in literal_eval(artifact_df[ch_name].iloc[ev_]):
        #             # remove 100 ms before 
        #             ev_ix_start = np.max([0, np.floor((artifact_- 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
        #             # remove 100 ms after
        #             ev_ix_end = np.min([mne_data_reref._data.shape[-1], np.ceil((artifact_ + 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
        #             mne_data_reref._data[ev_, ch_ix, ev_ix_start:ev_ix_end] = np.nan

                    
        # Get electrode df
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}/{day}_reref_elec_df')
        
        for region in ['HPC', 'AMY', 'OFC', 'dmPFC', 'dlPFC', 'ACC']:
            region_names = list(elec_df[elec_df.salman_region==region].label.values)
            if len(region_names)==0:
                continue
            FOOOFGroup_res, chan_df = analysis_utils.FOOOF_compute_epochs(mne_data_reref.copy().pick(region_names), 
                                                            tmin=analysis_evs['feedback_start'][0], 
                                                            tmax=analysis_evs['feedback_start'][1], 
                                                            **FOOOF_kwargs)

            chan_df['region'] = region
            chan_dfs.append(chan_df)

        chan_dfs = pd.concat(chan_dfs)
        chan_dfs['participant'] = subj
        chan_dfs['unique_label'] = chan_dfs['participant'] + '_' + chan_dfs['channel']
        chan_dfs.to_csv(f'{save_path}/{event}_FOOOF.csv', index=False)
        


In [ ]:
hpc_c = np.array([0.44314, 0.2902, 1])
amy_c = np.array([0.49412, 0, 0.90196])
ofc_c = np.array([1, 0.75686, 0.41569])
acc_c = np.array([0.7098, 0.27059, 0.68235])
dmpfc_c = np.array([0.40784, 0.01961, 0])
dlpfc_c = np.array([1, 0.06275, 0.06275])

In [ ]:
chan_dfs['log_freq'] = np.log10(chan_dfs['frequency']) 

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(10, 6))
sns.lineplot(data=chan_dfs[chan_dfs.frequency<50], x='frequency', y='PSD_raw', hue='region', 
             hue_order =  ['HPC', 'AMY', 'OFC', 'dmPFC', 'dlPFC', 'ACC'],
             palette= [hpc_c, amy_c, ofc_c, dmpfc_c, dlpfc_c, acc_c],
              errorbar='se')
sns.despine()

In [ ]:
# Load the OFC data for all subjects 
event = 'feedback_start'
region = 'HPC'
FOOOF_dfs = []
for subj in subj_ids:
    load_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj}'
    chan_df = pd.read_csv(f'{load_path}/{event}_FOOOF.csv')
    FOOOF_dfs.append(chan_df[chan_df.region==region])

FOOOF_dfs = pd.concat(FOOOF_dfs)

In [ ]:
plt.hist(FOOOF_dfs.loc[FOOOF_dfs.peak_freq>0, 'peak_freq'])

In [ ]:
sns.lineplot(data=FOOOF_dfs, x='frequency', y='PSD_corrected', errorbar='se', color='black')


In [ ]:
# Load the OFC rank data

rank_df = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/EMU/{region}_biomarker_rank.csv')

In [ ]:
FOOOF_dfs = FOOOF_dfs.merge(rank_df, on='unique_label', how='left')

In [ ]:
sns.regplot(FOOOF_dfs[FOOOF_dfs.peak_height!=0], x='overall_rank', y='peak_freq', scatter=False)

In [ ]:
sns.lineplot(data=FOOOF_dfs[FOOOF_dfs['overall_rank']<20], x='frequency', y='PSD_corrected', errorbar='se', color='black')
sns.lineplot(data=FOOOF_dfs[FOOOF_dfs['overall_rank']>20], x='frequency', y='PSD_corrected', errorbar='se')

In [ ]:
smf.mixedlm('PSD_exp ~ 1+overall_rank', 
            data=FOOOF_dfs, 
            groups=FOOOF_dfs['participant']).fit().summary()

In [ ]:
sns.regplot(data=FOOOF_dfs, x='overall_rank', y='PSD_exp', ci=95)
plt.ylim([0, 4])

In [ ]:
sns.lmplot(data=FOOOF_dfs, x='rank', y='PSD_exp', hue='participant', scatter=False)
plt.ylim([1, 4])

## Plot the power spectrum for a single trial

In [ ]:
def robust_regression_psd(freqs, psd, fmin, fmax):
    """
    a function to perform a robust regression for the log-log PSD, 
    using different end frequencies for the fit, and returning a slope 

    Parameters
    ----------
    freqs : array
        the frequency vector
    psd : array 
        the PSD vector
    fmin : float
        the minimum frequency for the fit
    fmax : float   
        the maximum frequency for the fit
    
    """

    # get the indices of the frequencies that are within the range of interest
    fmin_idx = np.argmin(np.abs(freqs - fmin))
    fmax_idx = np.argmin(np.abs(freqs - fmax))
    
    freqs = freqs[fmin_idx:fmax_idx]
    psd = psd[fmin_idx:fmax_idx]
    
    # fit a robust regression model to the log-log PSD
    model = sm.RLM(np.log10(psd), sm.add_constant(np.log10(freqs)))
    results = model.fit()
    
    # get the slope 
    slope = results.params[1]

    return slope

In [ ]:
def process_subject(subj_id, base_dir, day, event, ev_dict, robust_regression_psd, region_colors):
    load_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
    save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'

    # Load the data
    mne_data_reref = mne.read_epochs(f'{load_path}/{event}-epo.fif', preload=True)

    # Get electrode df
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

    slope_dfs = []

    for chan in mne_data_reref.ch_names:
        epo_spectrum = mne_data_reref.copy().pick([chan]).compute_psd(
            method='multitaper',
            fmin=2,
            fmax=200,
            tmin=ev_dict[event][0],
            tmax=ev_dict[event][-1],
            verbose=False
        )

        slopes = []
        for i in range(epo_spectrum.data.shape[0]):
            plt.plot(np.log10(epo_spectrum.freqs), np.log10(epo_spectrum.data[i, 0, :]), color = 'k', alpha = 0.1)
            slope = robust_regression_psd(epo_spectrum.freqs, epo_spectrum.data[i, 0, :], 2, 200)
            slopes.append(slope)

        region = elec_df[elec_df['label'] == chan].salman_region.values[0]
        color = region_colors.get(region, 'k')

        plt.plot(np.log10(epo_spectrum.freqs), np.log10(epo_spectrum.data.mean(axis=0)[0, :]), color=color, linewidth=2)
        

        plt.xticks(np.log10([2, 8, 13, 30, 70, 200]), [2, 8, 13, 30, 70, 200])
        plt.xlabel('Frequency (Hz)')
        plt.ylabel('Power (dB)')
        sns.despine()

        save_dir = f'{save_path}/PSD'
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig(f'{save_dir}/{chan}_PSD.pdf', dpi=300)
        plt.close()

        slope_df = pd.DataFrame({
            'slope': slopes,
            'trials': np.arange(epo_spectrum.data.shape[0]),
            'chan': chan,
            'participant': subj_id,
            'region': region
        })
        slope_dfs.append(slope_df)

    # Save this dataframe as CSV
    slope_dfs = pd.concat(slope_dfs)
    slope_dfs.to_csv(f'{save_path}/PSD/{event}_slopes.csv', index=False)

In [ ]:
def plot_psd_split_cond(subj_id, combined_df, base_dir, day, event, ev_dict, robust_regression_psd, region_colors):
    load_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
    save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'

    # Load the data
    mne_data_reref = mne.read_epochs(f'{load_path}/{event}-epo.fif', preload=True)

    # Get electrode df
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

    memory_df = combined_df[(combined_df.condition == 'Day1') & (combined_df.participant == subj_id)].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits', 'rpe']]
    memory_df['trials'] = memory_df['trials_gamble']


    for chan in mne_data_reref.ch_names:
        epo_spectrum = mne_data_reref.copy().pick([chan]).compute_psd(
            method='multitaper',
            fmin=2,
            fmax=200,
            tmin=ev_dict[event][0],
            tmax=ev_dict[event][-1],
            verbose=False
        )


        # slopes = []
        # for i in range(epo_spectrum.data.shape[0]):
        #     plt.plot(np.log10(epo_spectrum.freqs), np.log10(epo_spectrum.data[i, 0, :]), color = 'k', alpha = 0.1)
        #     slope = robust_regression_psd(epo_spectrum.freqs, epo_spectrum.data[i, 0, :], 2, 200)
        #     slopes.append(slope)

        region = elec_df[elec_df['label'] == chan].salman_region.values[0]
        color = region_colors.get(region, 'k')

        hit_pos_trials = memory_df.loc[(memory_df.hits == True) & (memory_df.rpe > 0), 'trials'].values.astype(int) - 1
        hit_neg_trials = memory_df.loc[(memory_df.hits == True) & (memory_df.rpe < 0), 'trials'].values.astype(int) - 1
        miss_pos_trials = memory_df.loc[(memory_df.hits == False) & (memory_df.rpe > 0), 'trials'].values.astype(int) - 1
        miss_neg_trials = memory_df.loc[(memory_df.hits == False) & (memory_df.rpe < 0), 'trials'].values.astype(int) - 1

        if len(hit_pos_trials) == 0 or len(hit_neg_trials) == 0 or len(miss_pos_trials) == 0 or len(miss_neg_trials) == 0:
            print('cant split the data as there are no trials in one of the conditions')
            continue

        # make a figure with two columns:
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))        
        ax[0].plot(np.log10(epo_spectrum.freqs), np.log10(epo_spectrum.data[miss_neg_trials].mean(axis=0)[0, :, ]), color=color, alpha=0.5, linewidth=2)
        ax[0].plot(np.log10(epo_spectrum.freqs), np.log10(epo_spectrum.data[hit_neg_trials].mean(axis=0)[0, :, ]), color=color, linewidth=2)
        ax[1].plot(np.log10(epo_spectrum.freqs), np.log10(epo_spectrum.data[miss_pos_trials].mean(axis=0)[0, :, ]), color=color, alpha=0.5, linewidth=2)
        ax[1].plot(np.log10(epo_spectrum.freqs), np.log10(epo_spectrum.data[hit_pos_trials].mean(axis=0)[0, :, ]), color=color, linewidth=2)
        

        ax[0].set_xticks(np.log10([2, 8, 13, 30, 70, 200]), [2, 8, 13, 30, 70, 200])
        ax[0].set_xlabel('Frequency (Hz)')
        ax[0].set_ylabel('Power (dB)')

        ax[1].set_xticks(np.log10([2, 8, 13, 30, 70, 200]), [2, 8, 13, 30, 70, 200])
        ax[1].set_xlabel('Frequency (Hz)')
        ax[1].set_ylabel('Power (dB)')

        sns.despine()

        save_dir = f'{save_path}/PSD'
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig(f'{save_dir}/{chan}_PSD_rpe_memory_split.pdf', dpi=300)
        plt.close()

        # slope_df = pd.DataFrame({
        #     'slope': slopes,
        #     'trials': np.arange(epo_spectrum.data.shape[0]),
        #     'chan': chan,
        #     'participant': subj_id,
        #     'region': region
        # })
        # slope_dfs.append(slope_df)

    # # Save this dataframe as CSV
    # slope_dfs = pd.concat(slope_dfs)
    # slope_dfs.to_csv(f'{save_path}/PSD/{event}_slopes.csv', index=False)

In [ ]:
# select event of interest
ev_dict = {'feedback_start': [0, 1.5]}

event = 'feedback_start'

# select day
day = 'Day1'

region_colors = {
    'HPC': hpc_c,
    'AMY': amy_c,
    'OFC': ofc_c,
    'ACC': acc_c,
    'dmPFC': dmpfc_c,
    'dlPFC': dlpfc_c,
}

# Parallelize using joblib
Parallel(n_jobs=-1)(
    delayed(plot_psd_split_cond)(
        subj_id,
        combined_df,
        base_dir,
        day,
        event,
        ev_dict,
        robust_regression_psd,
        region_colors
    )
    for subj_id in subj_ids
)

In [ ]:
# select event of interest
ev_dict = {'feedback_start': [0, 1.5]}

event = 'feedback_start'

# select day
day = 'Day1'

region_colors = {
    'HPC': hpc_c,
    'AMY': amy_c,
    'OFC': ofc_c,
    'ACC': acc_c,
    'dmPFC': dmpfc_c,
    'dlPFC': dlpfc_c,
}

# Parallelize using joblib
Parallel(n_jobs=-1)(
    delayed(process_subject)(
        subj_id,
        base_dir,
        day,
        event,
        ev_dict,
        robust_regression_psd,
        region_colors
    )
    for subj_id in subj_ids
)

In [ ]:



for subj_id in subj_ids:

    load_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
    save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'

    # load the data
    mne_data_reref = mne.read_epochs(f'{load_path}/{event}-epo.fif', preload=True)

    # Get electrode df 
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
            

    slope_dfs = []
    
    for chan in mne_data_reref.ch_names:

        epo_spectrum = mne_data_reref.copy().pick([chan]).compute_psd(method='multitaper',
                                    fmin=2, 
                                    fmax=200,
                                    tmin=ev_dict[event][0],
                                    tmax=ev_dict[event][-1],
                                    verbose=False)
        
        slopes = []
        for i in range(epo_spectrum.data.shape[0]):
            plt.plot(np.log10(epo_spectrum.freqs), np.log10(epo_spectrum.data[i, 0, :]), color = 'k', alpha = 0.1)
            plt.xticks(np.log10([2, 8, 13, 30, 70, 200]), [2, 8, 13, 30, 70, 200])
            slope = robust_regression_psd(epo_spectrum.freqs, epo_spectrum.data[i, 0, :], 2, 200)
            slopes.append(slope)

        if elec_df[elec_df['label'] == chan].salman_region.values[0] == 'HPC':
            color = hpc_c
        elif elec_df[elec_df['label'] == chan].salman_region.values[0] == 'AMY':
            color = amy_c
        elif elec_df[elec_df['label'] == chan].salman_region.values[0] == 'OFC':
            color = ofc_c
        elif elec_df[elec_df['label'] == chan].salman_region.values[0] == 'ACC':
            color = acc_c
        elif elec_df[elec_df['label'] == chan].salman_region.values[0] == 'dmPFC':
            color = dmpfc_c 
        elif elec_df[elec_df['label'] == chan].salman_region.values[0] == 'dlPFC':
            color = dlpfc_c
        else:
            color = 'k'

        plt.plot(np.log10(epo_spectrum.freqs), np.log10(epo_spectrum.data.mean(axis=0)[0, :]), color = color, linewidth = 2)
        plt.xlabel('Frequency (Hz)')
        plt.ylabel('Power (dB)')
        sns.despine()

        save_dir = f'{save_path}/PSD'
        if not os.path.exists(save_dir):
            os.makedirs(save_dir)
        plt.savefig(f'{save_dir}/{chan}_PSD.pdf', dpi=300)
        plt.close()

        slope_df = pd.DataFrame({'slope': slopes, 
                                'trials': np.arange(epo_spectrum.data.shape[0]), 
                                'chan': chan, 
                                'participant': subj_id, 
                                'region': elec_df[elec_df['label'] == chan].salman_region.values[0]})

        slope_dfs.append(slope_df)

    # save this dataframe out as csv
    slope_dfs = pd.concat(slope_dfs)
    slope_dfs.to_csv(f'{save_path}/PSD/{event}_slopes.csv', index=False)



In [ ]:
mne_data_reref.ch_names

In [ ]:
baseline_data_reref =  mne.read_epochs(f'{load_path}/baseline_start-epo.fif', preload=True)
mne_data_reref = mne.read_epochs(f'{load_path}/{event}-epo.fif', preload=True)

In [ ]:
chan = 'rmolf1-rmolf2'
event = 'feedback_start'
test_data = mne_data_reref.copy().crop(tmin=ev_dict[event][0], tmax=ev_dict[event][-1]).pick([chan])
# baseline_dat = baseline_data_reref.copy().crop(tmin=0, tmax=0.75).pick([chan])

In [ ]:
from scipy.signal import hilbert

trial = 4
window_size = 0.075
step_size = 0.025

baseline_channel = baseline_dat._data[trial, 0, :]

baseline_channel_filt = mne.filter.filter_data(baseline_channel, sfreq=mne_data_reref.info['sfreq'], l_freq=70, h_freq=200)
baseline_channel_amplitude = np.abs(hilbert(baseline_channel_filt)) ** 2
base_mean = np.mean(baseline_channel_amplitude)
base_std = np.std(baseline_channel_amplitude)

test_channel = test_data._data[trial, 0, :]

ts = np.arange(0, len(test_channel)/mne_data_reref.info['sfreq'], 1/mne_data_reref.info['sfreq'])
# filter the test channel between 70-200 Hz using mne
test_channel_filt = mne.filter.filter_data(test_channel, sfreq=mne_data_reref.info['sfreq'], l_freq=70, h_freq=200)
analytic_amplitude = ((np.abs(hilbert(test_channel_filt))** 2) - base_mean) / base_std

# smooth the signal using a moving average over 100 ms, with 25 ms steps
f, ax = plt.subplots(1, 1, figsize=(15, 5))


smoothed_amplitude = []
ts_smooth = [] # set the midpoint of the window as the timestamp
for i in np.arange(0, len(analytic_amplitude), int(step_size*mne_data_reref.info['sfreq'])):
    if i + int(window_size*mne_data_reref.info['sfreq']) < len(analytic_amplitude):
        smoothed_amplitude.append(np.mean(analytic_amplitude[i:i+int(window_size*mne_data_reref.info['sfreq'])]))
        ts_smooth.append(ts[i+int(window_size*mne_data_reref.info['sfreq'])//2])

plt.plot(ts_smooth, smoothed_amplitude, 'green')
plt.plot(ts, test_channel_filt, color='green', alpha=0.1)
plt.axhline(base_mean, color='red', linestyle='--')

# compute bursts as starts and stops that are 1.96 standard deviations above the mean
burst_starts = []
burst_stops = []
bursting = False
for i, amp in enumerate(analytic_amplitude):
    if amp > 1:
        if not bursting:
            burst_starts.append(ts[i])
            bursting = True
    else:
        if bursting:
            burst_stops.append(ts[i])
            bursting = False

# Each burst must be at least 25 ms long (or one bin) and no longer than 250 ms

min_threshold = (0.05 * mne_data_reref.info['sfreq']) / 1000
max_threshold = (0.50 * mne_data_reref.info['sfreq']) / 1000

burst_starts = [start for start, stop in zip(burst_starts, burst_stops) if stop - start > min_threshold]
burst_stops = [stop for start, stop in zip(burst_starts, burst_stops) if stop - start > min_threshold]
burst_starts = [start for start, stop in zip(burst_starts, burst_stops) if stop - start < max_threshold]
burst_stops = [stop for start, stop in zip(burst_starts, burst_stops) if stop - start < max_threshold]

# plot the bursts anove the signal
for start, stop in zip(burst_starts, burst_stops):
    plt.axvspan(start, stop, color='red', alpha=0.5)


# TODO: Save out the number of bursts, the % time bursting, the magnitude of the bursts, the timing of the bursts, for each trial in each channel. 
            
# TODO: Characterize sub-threshold tonic activity as well

In [ ]:
hfa_bursts = analysis_utils.detect_fast_burst_evs(mne_data_reref.crop(0, 1.5), 
                    baseline_data_reref.crop(tmin=0, tmax=0.75),
                    burst_frequency = (70, 200),
                    smooth_win_s = 0.05, 
                    sd_upper_cutoff=6, 
                    sd_lower_cutoff=0.1)

# ripple_evs = analysis_utils.detect_fast_burst_evs(mne_data_reref.crop(0, 1.5), 
#                     baseline_data_reref.crop(tmin=0, tmax=0.75),
#                     burst_frequency = (80, 120),
#                     smooth_win_s = 0.02, 
#                     sd_upper_cutoff=6, 
#                     sd_lower_cutoff=2.5)


In [ ]:
# get all the keys that are not empty in 'rmolf1-rmolf2' in ripple_evs
non_empty_trials = [k + 1 for k, v in hfa_bursts['rmolf1-rmolf2'].items() if len(v) > 0]

In [ ]:
hfa_bursts['rmolf1-rmolf2']

In [ ]:
# Step 1: band-pass filter the data
burst_frequency = (70, 200)
smooth_win_s = 0.05

filtered_data = mne_data.copy().crop(0, 1.5).filter(burst_frequency[0], burst_frequency[1], n_jobs=-1)

# Create an empty array to store the rolling RMS for each trial and time series
rolling_rms_array = np.zeros_like(filtered_data._data)

# Loop over each trial and each time series and calculate the rolling RMS
for i in range(filtered_data._data.shape[0]):
    for j in range(filtered_data._data.shape[1]):
        column_values = ['signal'] 
        df = pd.DataFrame(data = filtered_data._data[i, j, :], columns = column_values)
        smoothed_data = df['signal'].pow(2).rolling(round(smooth_win_s * mne_data.info['sfreq']), min_periods=1).mean().apply(np.sqrt)
        rolling_rms_array[i, j, :] = smoothed_data.values


In [ ]:
# plot the hfa burst using span for each trial and channel
ch_index = mne_data.ch_names.index('rmolf1-rmolf2')
for trial in [79]:
    plt.plot(rolling_rms_array[trial, ch_index, :], color='black')
    plt.plot(filtered_data._data[trial, ch_index, :], color='green', alpha=0.1)
    for start, stop, _ in hfa_bursts['rmolf1-rmolf2'][trial]:
        plt.axvspan(start, stop, color='red', alpha=0.5)

In [ ]:
ch_index = mne_data.ch_names.index('rmolf1-rmolf2')

In [ ]:
(0.25 * mne_data_reref.info['sfreq'])/1000

In [ ]:
max_threshold

In [ ]:
(burst_stops[0] -  burst_starts[0])

In [ ]:
(burst_stops[0] - burst_starts[0]) * 1000

In [ ]:
burst_starts

In [ ]:
sns.catplot(data=slope_dfs, x='region', y='slope', kind='box', aspect=2)

In [ ]:
plt.rcParams.update({
    'figure.dpi': 300,
    'figure.figsize': (8, 6),
    'lines.linewidth': 1.5,
    'lines.dash_capstyle': 'round',
    'font.family': 'Arial',
    'font.size': 24,
    'axes.labelsize': 26,
    'axes.titlesize': 28,
    'xtick.labelsize': 24,
    'ytick.labelsize': 24,
    'legend.fontsize': 24,
    'legend.frameon': False
})

for i in range(80):
    plt.plot(np.log10(epo_spectrum.freqs), np.log10(epo_spectrum.data[i, 0, :]), color = 'k', alpha = 0.1)
    # label x-ticks with the non-log values, specifically for the non-log values nearest to [2, 8, 13, 30, 70, 200]
    plt.xticks(np.log10([2, 8, 13, 30, 70, 200]), [2, 8, 13, 30, 70, 200])

# plot the mean PSD as well without transparency
plt.plot(np.log10(epo_spectrum.freqs), np.log10(epo_spectrum.data.mean(axis=0)[0, :]), color = 'r', linewidth = 2)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power (dB)')

sns.despine()


In [ ]:
slopes = [] 
for i in range(80):
    slope = robust_regression_psd(epo_spectrum.freqs, epo_spectrum.data[i, 0, :], 2, 200)
    slopes.append(slope)


In [ ]:
sns.regplot(x=learn_df[learn_df.participant==subj_id].rpe, y=slopes, color='k')

In [ ]:
plt.hist(slopes)

## Plot the power spec by region

Before using FOOOF to compute peak properties constrained to lower frequencies, let's first just plot the power spectra in a region-specific manner, normalizing for comparison across participants: 

In [ ]:
%%time
# select event of interest
ev_dict = {'feedback_start': [0, 1.5]}

# select ROI
rois = ['HPC', 'AMY', 'OFC', 'dmPFC', 'dlPFC', 'ACC'] 

progress_bar = tqdm(subj_ids, ascii=True, desc='Computing PSD for all trials and subsetting by conditions')

psd_group_data = {f'{a}': [] for a in rois}

for subj_id in progress_bar:
    # Load one person's data 
    for day in ['Day1']:
        load_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
        
        for event in ev_dict.keys():   
            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/FOOOF/{event}'
            if not os.path.exists(f'{filepath}/plots'):
                os.makedirs(f'{filepath}/plots')
            if not os.path.exists(f'{filepath}/dfs'):
                os.makedirs(f'{filepath}/dfs')

            mne_data_reref = mne.read_epochs(f'{load_path}/{event}-epo.fif', preload=True)
            mne_data_reref.metadata = learn_df[learn_df.participant==subj_id]
            
            mne_data_reref.crop(tmin=ev_dict[event][0], tmax=ev_dict[event][-1])
            
            # NAN out the bad data
            # THE following will now LOAD in dataframes that indicate IED and artifact time points in your data
            IED_df = pd.read_csv(f'{load_path}/{event}_IED_df.csv') 
            artifact_df = pd.read_csv(f'{load_path}/{event}_artifact_df.csv') 

            # Now, let's iterate through each channel, and each ied/artifact, and NaN 100 ms before and after these timepoints
            for ch_ix, ch_name in enumerate(mne_data_reref.ch_names): 
                ied_ev_list = IED_df[ch_name].dropna().index.tolist()
                artifact_ev_list = artifact_df[ch_name].dropna().index.tolist() 
                for ev_ in ied_ev_list: 
                    for ied_ in literal_eval(IED_df[ch_name].iloc[ev_]):
                        # remove 100 ms before 
                        ev_ix_start = np.max([0, np.floor((ied_- 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
                        # remove 100 ms after
                        ev_ix_end = np.min([mne_data_reref._data.shape[-1], np.ceil((ied_ + 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
                        mne_data_reref._data[ev_, ch_ix, ev_ix_start:ev_ix_end] = np.nan
                for ev_ in artifact_ev_list: 
                    for artifact_ in literal_eval(artifact_df[ch_name].iloc[ev_]):
                        # remove 100 ms before 
                        ev_ix_start = np.max([0, np.floor((artifact_- 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
                        # remove 100 ms after
                        ev_ix_end = np.min([mne_data_reref._data.shape[-1], np.ceil((artifact_ + 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
                        mne_data_reref._data[ev_, ch_ix, ev_ix_start:ev_ix_end] = np.nan


            save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
            # Get electrode df 
            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
                    
            progress_bar = tqdm(elec_df.label, ascii=True, desc='Computing PSDs')

            for elec_ix, chan in enumerate(progress_bar):
                region = elec_df.salman_region.iloc[elec_ix]
                if region in rois:
                    # compute PSD here 
                    epo_spectrum = mne_data_reref.copy().pick([chan]).compute_psd(method='multitaper',
                                                fmin=2, 
                                                fmax=50,
                                                tmin=ev_dict[event][0],
                                                tmax=ev_dict[event][-1],
                                                verbose=False)
                                                
#                     psds, freqs = epo_spectrum.get_data(return_freqs=True)
                    psds = epo_spectrum._data
                    freqs = epo_spectrum.freqs
                    # average across trials and squeeze out the empty axis 
                    psd_group_data[region].append(np.nanmean(psds, axis=0)[0,:])
                    
                    
                    
                    

the following function is just useful for normalizing the PSDs to the range of 0-1

In [ ]:
def normalize_list(input_list):
    min_val = np.nanmin(input_list)
    max_val = np.nanmax(input_list)
    den = (max_val - min_val)
    if den == 0: 
        den = 0.001
    
    normalized_list = [(x - min_val) / den  for x in input_list]
    
    return normalized_list

In [ ]:
# freqs = np.logspace(*np.log10([2, 200]), num=30)
save_dir_plots = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Plots'

for region in rois: 
    plot_freqs = freqs[freqs<40]
    fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=300)
    # Create a Pandas DataFrame
    plot_data = np.vstack(psd_group_data[region])
    normalized_array = np.array([normalize_list(lst) for lst in plot_data])
    for ch in normalized_array:
        # plt.plot(freqs, np.log10(ch), alpha=0.1, color='k')
        # plot_power_spectra(freqs, ch, alpha =0.1,  
        #                ax=ax, colors='black')   
        plt.plot(freqs[freqs<40], 
                 np.log10(ch)[freqs<40], alpha=0.05, color='k')

    # plot_power_spectra(freqs, np.nanmean(plot_data, axis=0), 
    #                    ax=ax, colors='black')   
    plt.plot(freqs[freqs<40], 
             np.log10(np.nanmean(normalized_array, axis=0))[freqs<40], alpha=1, color='k')
    

    n_elecs = len(psd_group_data[region])
    ax.set(xlabel='Frequency (Hz)', ylabel='Power (normalized)', title=f'{region}_nelec={n_elecs}', xticks=plot_freqs[::7].astype(int))

    sns.despine()
    
    # plt.savefig(f'{save_dir_plots}/feedback_PSD_{region}.pdf', dpi=300)

        
        

In [ ]:
freqs[freqs<50]

In [ ]:
normalized_array.shape

## Power spectrum (FOOOF)

Here, we are going to plot the group-level FOOOF analysis for two conditions in the task across all channels, and then also plot the individual power spectrums for each channel with model fit for each of the two conditions. Note the time specification to make sure I leave out the baseline period... 

FOOOF's primary utility, imo, is not in peak parameterization but 1/f subtraction. Peak parameters are subject to over- and under-fitting, and fit quality has too strong of an effect. Simply contrasting *n* PSDs (for *n* conditions) after removing the 1/f is a more healthy approach imo.

For guidance on picking the parameters used to fit: https://fooof-tools.github.io/fooof/auto_tutorials/plot_07-TroubleShooting.html#sphx-glr-auto-tutorials-plot-07-troubleshooting-py

## 1. FOOOF (produce dataframe and condition-specific plots)

In [ ]:
%%time
ev_dict = {'feedback_start': [0, 1.5]}

# Define peak_width, peak_height, peak_threshold, max_n_peaks
FOOOF_kwargs = {'peak_width_limits': [1, 8], 
               'min_peak_height': 0.0,
                'peak_threshold': 2.,
                'max_n_peaks': 3,
               'freq_range': [4, 30]}
 
# Define our frequency bands of interest
band_dict = {'theta' : [4, 8],
               'alpha' : [8, 13],
               'beta' : [13, 30]}

conditions = ['rpe>0',
             'rpe<1']

progress_bar = tqdm(subj_ids, ascii=True, desc='Computing FOOOF for all trials and subsetting by conditions')

for subj_id in progress_bar:
    # Load one person's data 
    for day in ['Day1']:
        load_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
        for event in ev_dict.keys():   
            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/FOOOF/{event}'
            if not os.path.exists(f'{filepath}/plots'):
                os.makedirs(f'{filepath}/plots')
            if not os.path.exists(f'{filepath}/dfs'):
                os.makedirs(f'{filepath}/dfs')

            mne_data_reref = mne.read_epochs(f'{load_path}/{event}-epo.fif', preload=True)
            
            # mne_data_reref.metadata = learn_df[learn_df.participant==subj_id]
            # mne_data_reref.metadata = s

            save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
            # Get electrode df 
            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
            
            # # Run the FOOOF code in parallel
            Parallel(n_jobs=-1, verbose=1)(delayed(analysis_utils.compute_FOOOF_parallel)(chan, mne_data_reref, 
                                                                                         subj_id, 
                                                                                         elec_df, 
                                                                                         event, 
                                                                                         ev_dict, 
                                                                                         band_dict, 
                                                                                         conditions, 
                                                                                         do_plot=True, 
                                                                                         save_path='/sc/arion/projects/guLab/Salman/EphysAnalyses',
                                                                                         do_save=True, 
                                                                                         **FOOOF_kwargs) for chan in mne_data_reref.ch_names)


In [ ]:
ev_dict = {'feedback_start': [0, 1.5]}


With 25 cores, this took ~0.5 second / electrode

In [ ]:
# Load the dataframes: 
chan_dfs = [] 
for subj_id in subj_ids[0:1]:
    for day in ['Day1']:
        load_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
        for event in ev_dict.keys():   
            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/FOOOF/{event}'
            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
            for chan in tqdm(elec_df.label.unique(), ascii=True, desc='Loading FOOOF dataframes'): 
                chan_df = pd.read_csv(f'{filepath}/dfs/{chan}_df.csv')
                chan_dfs.append(chan_df)
chan_dfs = pd.concat(chan_dfs)



In [ ]:
oneTauOU_oneF(theta, deltaT, binSize, T, numTrials, data_mean, data_var)



In [ ]:
# Label the frequency bands for easy indexing: 

chan_dfs['fband'] = chan_dfs.apply(lambda x: 'theta' if ((x.frequency>=2) & (x.frequency<8)) else 'alpha' if ((x.frequency>=8) & (x.frequency<13)) else 'beta' if ((x.frequency>=13) & (x.frequency<=30)) else np.nan, axis=1)



Here I want to identify all the "theta electrodes" - electrodes with significant peaks in the theta range

In [ ]:
roi_df = chan_dfs[chan_dfs['region'].isin(['HPC', 'AMY', 'OFC'])]
roi_df = roi_df[(roi_df.cond=='all')]
roi_df['theta_peak'] = 0

In [ ]:
# these are the electrodes with >= 1 peak in theta frequencies
theta_channel_ids = roi_df[((roi_df['peak_freq'] > 2 ) & (roi_df['peak_freq'] < 8))][['participant', 'channel']].drop_duplicates()

for row in theta_channel_ids.itertuples():
    roi_df.loc[((roi_df.participant==row.participant) & (roi_df.channel==row.channel)), 'theta_peak'] = 1

In [ ]:
# roi_df['log_psd'] = np.log10(roi_df['PSD_raw'])
# roi_df['log_freq'] = np.log10(roi_df['frequency'])

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4), dpi=300, sharey=True)

sns.lineplot(data=roi_df[roi_df.region=='AMY'], 
             x='frequency', 
             y='PSD_raw', 
             errorbar='se', 
             hue='theta_peak', 
             ax=ax[0])
sns.lineplot(data=roi_df[roi_df.region=='HPC'], 
             x='frequency', 
             y='PSD_raw', 
             errorbar='se', 
             hue='theta_peak', 
             ax=ax[1])
sns.lineplot(data=roi_df[roi_df.region=='OFC'], 
             x='frequency', 
             y='PSD_raw', 
             errorbar='se', 
             hue='theta_peak', 
             ax=ax[2])

fig.tight_layout()
sns.despine()


In [ ]:
roi_df[['participant', 'region']].drop_duplicates().shape


In [ ]:
roi_df[((roi_df['peak_freq'] > 2 ) & (roi_df['peak_freq'] < 8))][['participant', 'region']].drop_duplicates().shape



In [ ]:
theta_channel_ids

In [ ]:
theta_channel_ids.to_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/theta_channel_df', index=None)


In [ ]:
for col in ['in_FOOOF_peak', 'peak_freq', 'peak_height']:
    roi_df[col] = roi_df[col].apply(lambda x: np.nan if x==0 else x) 


In [ ]:
plot_df = roi_df[(roi_df.fband=='theta') & (roi_df.theta_peak==1)].groupby(['participant', 'channel', 'region']).mean().reset_index()

plot_df


In [ ]:
ax = sns.jointplot(data = plot_df, 
                   x='PSD_exp', y='peak_freq', height=8, hue='region', ratio=2)
 
# plt.subplots_adjust(left=0.1, right=0.8, top=0.9, bottom=0.1)
# # get the current positions of the joint ax and the ax for the marginal x
# pos_joint_ax = ax.ax_joint.get_position()
# pos_marg_x_ax = ax.ax_marg_x.get_position()
# # reposition the joint ax so it has the same width as the marginal x ax
# ax.ax_joint.set_position([pos_joint_ax.x0, pos_joint_ax.y0, pos_marg_x_ax.width, pos_joint_ax.height])
# # reposition the colorbar using new x positions and y positions of the joint ax
# ax.fig.axes[-1].set_position([.83, pos_joint_ax.y0, .07, pos_joint_ax.height])

In [ ]:
grouped_df = chan_dfs.groupby(['region', 'channel', 'cond', 'in_FOOOF_peak']).mean().reset_index().drop(columns=['in_FOOOF_peak', 'frequency', 'PSD_raw'])


                                                                                 
                                                                                 

In [ ]:
grouped_df[grouped_df.peak_freq!=0]

In [ ]:
# # Fit an individual channel (for plotting)
# ind_fits = FOOOFGroup_res.get_fooof(ind=0, regenerate=True)
# ind_fits.fit()

In [ ]:
# ind_fits.plot()